## EA Pipeline Skeleton

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import datetime
import random
import sys
import os

### PlaneModel defines a set of useful classes

In [ ]:
from  PlaneModel import job,aircraft,staff,problem

### Import Data Generator and its aircraft instaces method

In [27]:
from generate_aircrafts_seed import generate_aircraft_instances

### Import EA Computing Functions and Depencies

In [ ]:
from EA_Model import randSol, evaluate, timeMutate, mutate, copyG, contains, xo, tour, rip

### Generate CSV file with specified input parameters

In [ ]:
generate_aircraft_instances(
    work_packages_file="work_packages.csv",  # Relative Path to WP input CSV
    output_dir="generated_aircrafts",        # Output directory name
    num_instances=10,                        # Number of CSV files to generate
    seed=20                                  # Fix the seed to 20 to ensure reproducibility
)

### Conditional Checking of Data Directory

In [ ]:
def is_directory_empty(path_directory):
    """
    Check if a directory is empty.
    """
    path = Path(path_directory)
    # Check if the path exists and is a directory
    if not path.exists() or not path.is_dir():
        raise ValueError("Invalid directory path")
    # check if the directory is empty
    return not any(path.iterdir())

In [ ]:
path_directory = "generated_aircrafts"
try:
    if is_directory_empty(path_directory):
        print("Directory is empty")
    else:
        print("sample aircrfat instances are generated")
except ValueError as e:
    print(e)
    sys.exit(1)
    

### Reading Work Packages and Technicians files to DF

In [ ]:
work_packages = pd.read_csv('work_packages.csv')
people = pd.read_csv('technicians.csv')

## Load Problem instance from generator output directory


In [ ]:
file = './Update-Test-Data/ExampleUpdated10.xlsx'
planes = pd.read_excel(file,  sheet_name = 0)

### Populate Model



In [ ]:
instance = problem(people,planes,work_packages)

# EA

In [ ]:

          
pop_size = 1500 
budget = 100000

best = None

pop = []
for c in range(0,pop_size):

    i = randSol()
    f= evaluate(i)[0]
    p= (f,i)
    pop.append(p)
    if len(pop)==1:
        best=p
    
    
    if f < best[0]:
        best=p
        
print("Init")
print(best[0])
evals =0
while (evals < budget):
    evals = evals +1
    if random.choice([True,False]):
        parent = tour(pop)
        ng = copyG(parent[1])
    else:
        ng = xo(tour(pop)[1],tour(pop)[1])
        
    mutate(ng)
    nf = evaluate(ng)[0]
    child = (nf,ng)

    toGo = rip(pop)
    if toGo[0] > child[0]:
        pop.remove(toGo)
        pop.append(child)
        if child[0] < best[0]:
            best = (child[0],copyG(child[1])) 
            instance.reset()
            r = evaluate(best[1])
            print("Evals: " + str(evals))
            print("Improved (" +str(r[0])+") Missing staff = "+ str(r[1]) + " Late = "+ str(r[2]) + " Late mins = " + str(r[3]))
              
print("Done :" +str(best[0]))


In [ ]:
print("Final Result:")
instance.reset()
r = evaluate(best[1])
print("Fitness = " +str(r[0])+" Missing staff = "+ str(r[1]) + " Late Departures= "+ str(r[2]) + " Late mins = " + str(r[3]))
print("\nPlan:\n\n"+str(instance))

              

### Fine-tuning the EA outputs

In [ ]:
fitness_All = []
for run in range(10):
    pop_size = 1500 
    budget = 100000

    best = None

    pop = []
    for c in range(pop_size):
        i = randSol()
        f = evaluate(i)[0]
        p = (f, i)
        pop.append(p)
        if len(pop) == 1:
            best = p
        if f < best[0]:
            best = p

    print(f"Run {run + 1} - Init")
    print(f"Initial best: {best[0]}")

    evals = 0
    
    while evals < budget:
        evals += 1
        if random.choice([True, False]):
            parent = tour(pop)
            ng = copyG(parent[1])
        else:
            ng = xo(tour(pop)[1], tour(pop)[1])
        
        mutate(ng)
        nf = evaluate(ng)[0]
        child = (nf, ng)

        toGo = rip(pop)
        if toGo[0] > child[0]:
            pop.remove(toGo)
            pop.append(child)
            if child[0] < best[0]:
                best = (child[0], copyG(child[1]))
                instance.reset()
                r = evaluate(best[1])
                print(f"Evals: {evals}")
                print(f"Improved ({r[0]}) Missing staff = {r[1]} Late = {r[2]} Late mins = {r[3]}")

    fitness_All.append(best[0])
    print(f"Run {run + 1} - Done: {best[0]}\n")
    
# compute the average of fitness values in the population for 10 iterations
np_avg_fitness = np.mean(fitness_All)
avg_fitness = sum(fitness_All) / len(fitness_All)
print(f"Average fitness over 10 runs: Math Method {avg_fitness} and Numpy Method {np_avg_fitness}")

# compute standard deviation of fitness values in the population for 10 iterations
std_dev = np.std(fitness_All)
print(f"Standard Deviation: {std_dev}")